# [2장 2강] 실습문제 - 결측치 처리

## 실습 목표

- 컬럼별 결측치 개수와 비율을 진단할 수 있습니다.
- `dropna()`와 `fillna()`를 데이터 특성에 맞게 적용할 수 있습니다.
- 수치형과 범주형 결측치의 처리 방법 차이를 설명할 수 있습니다.
- 결측치 처리 방법과 선택 근거를 코드 주석과 정책표로 기록할 수 있습니다.

## 중요

이번 강은 **결측치 처리만** 다룹니다.

- 이상치 탐지 X
- 이상치 삭제 X
- 이상치 수정 X
- 이상치 대체 X

이상치 처리는 이후 강의에서 별도로 진행합니다.

## 사용 데이터

- 원본 파일: `Medical_Appoingtment_No_Shows.csv`

원본 데이터에는 결측치가 없으므로,
실습 시작 시 원본을 `copy()`한 뒤 **실습용 복사본 `df`에만 결측치를 의도적으로 생성**합니다.

원본 `appointments`는 수정하지 않습니다.

## 실습 준비

아래 코드를 그대로 실행하세요.

> 이번 셀은 결측치 처리 문법을 연습하기 위한 **실습 환경 준비 코드**입니다.

In [1]:
import pandas as pd
import numpy as np

# 원본 데이터 불러오기
appointments = pd.read_csv("Medical_Appoingtment_No_Shows.csv")

# 원본은 보존하고 실습용 복사본 생성
df = appointments.copy()

# 재현 가능한 실습을 위해 난수 고정
rng = np.random.default_rng(42)

# 실습용 결측치 생성
n = len(df)

age_idx = rng.choice(
    df.index,
    size=int(n * 0.04),
    replace=False
)

remaining_idx = df.index.difference(age_idx)

neighbourhood_idx = rng.choice(
    remaining_idx,
    size=int(n * 0.08),
    replace=False
)

remaining_idx = remaining_idx.difference(neighbourhood_idx)

sms_idx = rng.choice(
    remaining_idx,
    size=int(n * 0.05),
    replace=False
)

df.loc[age_idx, "Age"] = np.nan
df.loc[neighbourhood_idx, "Neighbourhood"] = np.nan
df.loc[sms_idx, "SMS_received"] = np.nan

print("원본 전체 결측치:", appointments.isna().sum().sum())
print("실습용 전체 결측치:", df.isna().sum().sum())

원본 전체 결측치: 0
실습용 전체 결측치: 18789


---

## 필수 1. 결측치 위치와 비율 진단하기

### 문제 1-1. 컬럼별 결측 요약표 만들기

**요구사항**

1. `isna().sum()`으로 컬럼별 결측 개수를 계산하세요.
2. 전체 행 수를 이용해 결측 비율(%)을 계산하세요.
3. `missing_count`, `missing_ratio` 컬럼을 가진 DataFrame을 만드세요.
4. 결측 비율이 높은 컬럼부터 내림차순 정렬하세요.
5. 결측치가 있는 컬럼만 확인하세요.
6. 결측 비율만으로 처리 방법을 결정하면 안 되는 이유를 작성하세요.

In [ ]:
# 문제 1-1 코드를 작성하세요.

missing_count = df.isna().sum()

total_rows = len(df)
missing_ratio = (missing_count / len(df)*100).round(2)

missing_df = (pd.DataFrame({"missing_count": missing_count,"missing_ratio": missing_ratio})
              .sort_values("missing_ratio",ascending=False))

print(missing_df)
print(missing_df[missing_df["missing_count"] > 0])



                missing_count  missing_ratio
Neighbourhood            8842            8.0
SMS_received             5526            5.0
Age                      4421            4.0
Gender                      0            0.0
AppointmentID               0            0.0
PatientId                   0            0.0
AppointmentDay              0            0.0
ScheduledDay                0            0.0
Hipertension                0            0.0
Scholarship                 0            0.0
Diabetes                    0            0.0
Alcoholism                  0            0.0
Handcap                     0            0.0
No-show                     0            0.0
               missing_count  missing_ratio
Neighbourhood           8842            8.0
SMS_received            5526            5.0
Age                     4421            4.0


### 문제 1-1 결과 해석

- 결측치가 있는 컬럼:
- 결측 비율이 가장 높은 컬럼:
- 결측 비율만으로 처리 방법을 결정하면 안 되는 이유:

---

## 필수 2. `dropna()`로 행 삭제 영향 확인하기

### 문제 2-1. 전체 삭제와 특정 컬럼 기준 삭제 비교

**요구사항**

1. 삭제 전 전체 행 수를 확인하세요.
2. `df.dropna()`로 결측치가 하나라도 있는 행을 모두 삭제하세요.
3. 삭제 후 행 수, 삭제된 행 수, 손실률을 계산하세요.
4. `df.dropna(subset=["Age"])`를 적용하세요.
5. 두 방식의 행 수 차이를 비교하세요.
6. 전체 삭제와 특정 컬럼 기준 삭제의 차이를 설명하세요.

In [ ]:
# 문제 2-1 코드를 작성하세요.

missing_count= df.isna().sum()

drop_df = df.dropna()

print('삭제 전:', len(df))
print('삭제 후: ', len(drop_df))

drop_age = df.dropna(subset=["Age"])
print("Age 결측만 삭제 후:", len(drop_age))


삭제 전: 110527
삭제 후:  91738
Age 결측만 삭제 후: 106106


### 문제 2-1 결과 해석

- 삭제 전 행 수:
- 전체 결측 행 삭제 후 행 수:
- 전체 삭제 손실률:
- Age 기준 삭제 후 행 수:
- 두 방식의 차이:

---

## 필수 3. 범주형 결측치 `fillna()` 처리하기

### 문제 3-1. `Neighbourhood` 결측치 처리

**요구사항**

1. `df.copy()`로 `filled_category`를 만드세요.
2. `Neighbourhood` 처리 전 결측 개수를 확인하세요.
3. `mode()[0]`으로 최빈값을 확인하세요.
4. 아래 두 방법 중 하나를 선택해 처리하세요.
   - 최빈값 대체
   - `"Unknown"` 별도 범주 대체
5. `fillna()`를 사용하세요.
6. 처리 후 결측 개수를 확인하세요.
7. 선택 근거를 코드 주석으로 남기세요.

In [15]:
# 문제 3-1 코드를 작성하세요.

filled_category =df.copy()
print("처리 전:",len(filled_category))

print("처리 후",filled_category["Neighbourhood"].isna().sum())

mode_value = filled_category["Neighbourhood"].mode()[0]

print("\nNeighbourhood 최빈값:")
print(mode_value)

filled_category["Neighbourhood"] = filled_category["Neighbourhood"].fillna(mode_value)

print("\n처리 후 결측 개수:")
print(filled_category["Neighbourhood"].isna().sum())


처리 전: 110527
처리 후 8842

Neighbourhood 최빈값:
JARDIM CAMBURI

처리 후 결측 개수:
0


### 문제 3-1 결과 해석

- 처리 전 결측 개수:
- 최빈값:
- 선택한 처리 방법:
- 처리 후 결측 개수:
- 선택 근거:

---

## 필수 4. 수치형 결측치 `fillna()` 처리하기

### 문제 4-1. `Age` 평균·중앙값 대체 비교

이번 문제에서는 **이상치 분석을 하지 않습니다.**

수치형 결측치 처리 방법으로 평균과 중앙값을 각각 적용해보고 차이를 비교합니다.

**요구사항**

1. `Age.mean()`과 `Age.median()`을 계산하세요.
2. `df.copy()`를 이용해 다음 DataFrame을 만드세요.
   - `age_mean_filled`
   - `age_median_filled`
3. `age_mean_filled`의 `Age` 결측치를 평균으로 대체하세요.
4. `age_median_filled`의 `Age` 결측치를 중앙값으로 대체하세요.
5. 처리 후 결측 개수를 확인하세요.
6. 평균 대체와 중앙값 대체의 차이를 설명하세요.
7. 실제 업무에서는 결측 원인과 데이터 특성을 확인한 뒤 방법을 결정해야 하는 이유를 작성하세요.

In [ ]:
# 문제 4-1 코드를 작성하세요.

age_mean = df["Age"].mean()
age_median = df["Age"].median()

age_mean_filled = df.copy()
age_median_filled = df.copy()

age_mean_filled["Age"] = age_mean_filled["Age"].fillna(age_mean)

age_median_filled["Age"] = age_median_filled["Age"].fillna(age_median)

print("\n평균 대체 후 Age 결측 개수:")
print(age_mean_filled["Age"].isna().sum())

print("\n중앙값 대체 후 Age 결측 개수:")
print(age_median_filled["Age"].isna().sum())

Age 평균: 37.079910655382356
Age 중앙값: 37.0

평균 대체 후 Age 결측 개수:
0

중앙값 대체 후 Age 결측 개수:
0


### 문제 4-1 결과 해석

- Age 평균:
- Age 중앙값:
- 평균 대체 후 결측 개수:
- 중앙값 대체 후 결측 개수:
- 두 방식의 차이:
- 실제 처리 방법을 바로 결정하면 안 되는 이유:

---

## 필수 5. 이진형 결측치 처리하기

### 문제 5-1. `SMS_received` 결측치 처리

`SMS_received`는 0과 1로 구성된 값입니다.

**요구사항**

1. `SMS_received`의 결측 개수를 확인하세요.
2. 결측치를 제외한 값의 빈도를 확인하세요.
3. 최빈값을 확인하세요.
4. `fillna()`로 최빈값 대체를 적용하세요.
5. 처리 후 결측 개수를 확인하세요.
6. 최빈값 대체의 장점과 주의점을 작성하세요.

In [17]:
# 문제 5-1 코드를 작성하세요.

missing_count = appointments["SMS_received"].isna().sum()
print("처리 전 결측 개수:", missing_count)

value_counts = appointments["SMS_received"].dropna().value_counts()
print("\n결측치를 제외한 값의 빈도:")
print(value_counts)

mode_value = appointments["SMS_received"].mode()[0]
print("\n최빈값:", mode_value)

appointments["SMS_received"] = appointments["SMS_received"].fillna(mode_value)

print("\n처리 후 결측 개수:", appointments["SMS_received"].isna().sum())

처리 전 결측 개수: 0

결측치를 제외한 값의 빈도:
SMS_received
0    75045
1    35482
Name: count, dtype: int64

최빈값: 0

처리 후 결측 개수: 0


---

## 심화 1. 결측치 처리 정책표 작성하기

### 문제 6-1

이번 실습의 세 컬럼을 대상으로 정책표를 완성하세요.

| 컬럼 | 데이터 유형 | 처리 방법 | 선택 근거 | 데이터 손실 여부 | 주의점 |
|---|---|---|---|---|---|
| Age |  |  |  |  |  |
| Neighbourhood |  |  |  |  |  |
| SMS_received |  |  |  |  |  |

**작성 기준**

- 단순히 `fillna()`를 썼다고 적지 말고 왜 그 방법을 선택했는지 작성하세요.
- 결측 처리로 분포나 비율이 달라질 가능성도 기록하세요.

---

# 실습 마무리

1. `dropna()`와 `fillna()`의 가장 큰 차이는 무엇인가요?
2. `dropna(subset=[...])`는 언제 유용한가요?
3. 범주형 결측치를 최빈값으로 대체할 때 생길 수 있는 문제는 무엇인가요?
4. 평균 대체와 중앙값 대체의 차이는 무엇인가요?
5. 결측치 처리 근거를 기록해야 하는 이유는 무엇인가요?
6. 원본 데이터를 직접 수정하지 않고 복사본으로 실습한 이유는 무엇인가요?